
# SmolLM: Transformer Implementation from Scratch

## Objective
Implement a decoder-only causal language model using PyTorch.

## Architecture
1. Configuration
2. RMSNorm
3. Rotary Positional Embeddings (RoPE)
4. Grouped-Query Attention (GQA)
5. SwiGLU feed-forward network
6. Pre-normalized decoder block
7. Full causal language model with tied embeddings

## Validation strategy
- Validate tensor shapes at each stage.
- Compare mathematical operations with reference implementations.
- Check causal masking and numerical stability.
- Verify parameter count and tied weights.
- Later, load pretrained checkpoint weights and compare logits.


In [60]:

import json
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoConfig

SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cpu
Device: cpu


In [61]:
assert isinstance(DEVICE, torch.device)
assert torch.get_num_threads() >= 1

print("Environment checks passed.")

Environment checks passed.


In [62]:

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

hf_config = AutoConfig.from_pretrained(MODEL_ID)

print("Model type:", hf_config.model_type)
print("Hidden size:", hf_config.hidden_size)
print("Layers:", hf_config.num_hidden_layers)
print("Attention heads:", hf_config.num_attention_heads)
print("KV heads:", hf_config.num_key_value_heads)
print("Intermediate size:", hf_config.intermediate_size)
print("Vocabulary size:", hf_config.vocab_size)
print("Tied embeddings:", hf_config.tie_word_embeddings)


Model type: llama
Hidden size: 576
Layers: 30
Attention heads: 9
KV heads: 3
Intermediate size: 1536
Vocabulary size: 49152
Tied embeddings: True


In [63]:

@dataclass
class SmolLMConfig:
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: int
    max_position_embeddings: int
    rms_norm_eps: float
    rope_theta: float
    tie_word_embeddings: bool
    attention_bias: bool = False
    mlp_bias: bool = False
    attention_dropout: float = 0.0

    @property
    def head_dim(self) -> int:
        assert self.hidden_size % self.num_attention_heads == 0
        return self.hidden_size // self.num_attention_heads

    def validate(self):
        assert self.vocab_size > 0
        assert self.hidden_size > 0
        assert self.num_hidden_layers > 0
        assert self.num_attention_heads > 0
        assert self.num_key_value_heads > 0

        assert self.hidden_size % self.num_attention_heads == 0
        assert self.num_attention_heads % self.num_key_value_heads == 0

        assert self.head_dim % 2 == 0
        assert self.intermediate_size > 0
        assert self.max_position_embeddings > 0
        assert self.rms_norm_eps > 0
        assert self.rope_theta > 0


config = SmolLMConfig(
    vocab_size=hf_config.vocab_size,
    hidden_size=hf_config.hidden_size,
    intermediate_size=hf_config.intermediate_size,
    num_hidden_layers=hf_config.num_hidden_layers,
    num_attention_heads=hf_config.num_attention_heads,
    num_key_value_heads=hf_config.num_key_value_heads,
    max_position_embeddings=hf_config.max_position_embeddings,
    rope_theta=getattr(hf_config, "rope_theta", 100000.0),
    rms_norm_eps=hf_config.rms_norm_eps,
    tie_word_embeddings=hf_config.tie_word_embeddings,
    attention_bias=hf_config.attention_bias,
    mlp_bias=getattr(hf_config, "mlp_bias", False),
    attention_dropout=hf_config.attention_dropout,
)

config.validate()
print(config)
print("Head dimension:", config.head_dim)


SmolLMConfig(vocab_size=49152, hidden_size=576, intermediate_size=1536, num_hidden_layers=30, num_attention_heads=9, num_key_value_heads=3, max_position_embeddings=8192, rms_norm_eps=1e-05, rope_theta=100000.0, tie_word_embeddings=True, attention_bias=False, mlp_bias=False, attention_dropout=0.0)
Head dimension: 64


In [64]:
assert config.hidden_size == 576
assert config.num_hidden_layers == 30
assert config.num_attention_heads == 9
assert config.num_key_value_heads == 3
assert config.head_dim == 64

print(config.rope_theta)

print("A1 passed.")

100000.0
A1 passed.



## A2 — RMSNorm

RMSNorm normalizes the magnitude of each token's hidden-state vector.

For input shape `(batch_size, sequence_length, hidden_size)`, it computes the
root mean square across the final dimension, then applies a learnable weight.

We accumulate the squared values in float32 for numerical stability, then
return the normalized result in the original input dtype.

Analogy: RMSNorm is like an automatic volume controller for each token's
feature vector. It controls the overall magnitude while preserving the
relative proportions of its features, before applying learned scaling.


In [65]:

class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype

        x = x.float()
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

        return (x * rms * self.weight.float()).to(input_dtype)

In [66]:
class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim: int,
        max_position_embeddings: int,
        theta: float = 100_000.0,
    ):
        super().__init__()

        assert head_dim % 2 == 0

        self.head_dim = head_dim
        self.theta = theta

        # Each pair of features uses its own rotation frequency.
        inv_freq = 1.0 / (
            theta ** (
                torch.arange(0, head_dim, 2, dtype=torch.float32)
                / head_dim
            )
        )

        self.register_buffer("inv_freq", inv_freq, persistent=False)

        positions = torch.arange(
            max_position_embeddings, dtype=torch.float32
        )

        freqs = torch.outer(positions, inv_freq)

        # Repeat each frequency for its corresponding feature pair.
        emb = torch.cat((freqs, freqs), dim=-1)

        self.register_buffer(
            "cos_cached", emb.cos(), persistent=False
        )
        self.register_buffer(
            "sin_cached", emb.sin(), persistent=False
        )

    def forward(self, x: torch.Tensor, position_ids=None):
        # x shape: (batch, heads, sequence_length, head_dim)
        seq_len = x.shape[-2]

        if position_ids is None:
            cos = self.cos_cached[:seq_len]
            sin = self.sin_cached[:seq_len]
        else:
            cos = self.cos_cached[position_ids]
            sin = self.sin_cached[position_ids]

        return cos, sin


In [67]:

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]

    return torch.cat((-x2, x1), dim=-1)


def apply_rope(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
):
    # Add batch/head broadcast dimensions to cos and sin.
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    cos = cos.to(dtype=q.dtype, device=q.device)
    sin = sin.to(dtype=q.dtype, device=q.device)

    q_rotated = q * cos + rotate_half(q) * sin
    k_rotated = k * cos + rotate_half(k) * sin

    return q_rotated, k_rotated


In [68]:

class GroupedQueryAttention(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.attention_dropout = config.attention_dropout

        # Query has more heads than key and value.
        self.q_proj = nn.Linear(
            self.hidden_size,
            self.num_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.k_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.v_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=config.attention_bias,
        )

        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim,
            self.hidden_size,
            bias=config.attention_bias,
        )

        self.rotary_emb = RotaryEmbedding(
            head_dim=self.head_dim,
            max_position_embeddings=config.max_position_embeddings,
            theta=config.rope_theta,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape

        # Project hidden states into Q, K and V.
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # (batch, seq_len, heads * head_dim)
        # -> (batch, heads, seq_len, head_dim)
        q = q.view(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        # Apply positional rotations to Q and K.
        cos, sin = self.rotary_emb(q)
        q, k = apply_rope(q, k, cos, sin)

        # Expand KV heads to match the query-head groups.
        k = k.repeat_interleave(self.num_kv_groups, dim=1)
        v = v.repeat_interleave(self.num_kv_groups, dim=1)

        # Causal scaled dot-product attention.
        attn_output = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=(
                self.attention_dropout if self.training else 0.0
            ),
            is_causal=True,
        )

        # (batch, heads, seq_len, head_dim)
        # -> (batch, seq_len, hidden_size)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(
            batch_size, seq_len, self.hidden_size
        )

        return self.o_proj(attn_output)


In [69]:
class SwiGLU(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.gate_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=config.mlp_bias,
        )

        self.up_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=config.mlp_bias,
        )

        self.down_proj = nn.Linear(
            config.intermediate_size,
            config.hidden_size,
            bias=config.mlp_bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)

        hidden = gate * up

        return self.down_proj(hidden)


In [70]:
class DecoderBlock(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.input_layernorm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.self_attn = GroupedQueryAttention(config)

        self.post_attention_layernorm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.mlp = SwiGLU(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Attention sublayer + residual connection
        x = x + self.self_attn(self.input_layernorm(x))

        # MLP sublayer + residual connection
        x = x + self.mlp(self.post_attention_layernorm(x))

        return x

In [71]:

class SmolLMForCausalLM(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.config = config

        # 1. Convert token IDs into hidden vectors
        self.embed_tokens = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
        )

        # 2. Stack independent Transformer blocks
        self.layers = nn.ModuleList([
                DecoderBlock(config)
            for _ in range(config.num_hidden_layers)
        ])

        # 3. Normalize the final hidden representations
        self.norm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        # 4. Project hidden vectors to vocabulary logits
        self.lm_head = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False,
        )

        # 5. Share input embedding and output projection weights
        if config.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # input_ids: (batch_size, sequence_length)
        x = self.embed_tokens(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        logits = self.lm_head(x)

        # logits: (batch_size, sequence_length, vocab_size)
        return logits


In [72]:
from transformers import AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
hf_model.eval()

print(type(hf_model).__name__)
print("Reference parameters:", sum(p.numel() for p in hf_model.parameters()))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LlamaForCausalLM
Reference parameters: 134515008


In [73]:

def load_hf_weights(
    model: SmolLMForCausalLM,
    hf_model: nn.Module,
):
    source_state = hf_model.state_dict()
    target_state = model.state_dict()

    mapped_state = {}
    used_source_keys = set()
    ignored_source_keys = set()

    for source_key, tensor in source_state.items():
        # Our RoPE computes these buffers locally.
        if source_key.endswith(
            ("rotary_emb.inv_freq",
             "rotary_emb.cos_cached",
             "rotary_emb.sin_cached")
        ):
            ignored_source_keys.add(source_key)
            continue

        if source_key.startswith("model.layers."):
            target_key = source_key.removeprefix("model.")
        elif source_key in (
            "model.embed_tokens.weight",
            "model.norm.weight",
        ):
            target_key = source_key.removeprefix("model.")
        elif source_key.startswith("lm_head."):
            target_key = source_key
        else:
            raise KeyError(
                f"Unrecognized HF parameter: {source_key}"
            )

        if target_key not in target_state:
            raise KeyError(
                f"No destination for {source_key} -> {target_key}"
            )

        if target_key in mapped_state:
            raise KeyError(f"Duplicate mapping: {target_key}")

        if tensor.shape != target_state[target_key].shape:
            raise ValueError(
                f"Shape mismatch for {target_key}: "
                f"HF={tuple(tensor.shape)}, "
                f"ours={tuple(target_state[target_key].shape)}"
            )

        mapped_state[target_key] = tensor
        used_source_keys.add(source_key)

    # If the checkpoint omits a tied output-head entry,
    # reuse its input embedding tensor.
    if model.config.tie_word_embeddings:
        embedding_key = "embed_tokens.weight"
        head_key = "lm_head.weight"

        if head_key not in mapped_state:
            mapped_state[head_key] = mapped_state[embedding_key]

    missing_keys = set(target_state) - set(mapped_state)
    if missing_keys:
        raise KeyError(f"Missing destination weights: {sorted(missing_keys)}")

    unexpected_keys = (
        set(source_state) - used_source_keys - ignored_source_keys
    )
    if unexpected_keys:
        raise KeyError(f"Unused HF weights: {sorted(unexpected_keys)}")

    model.load_state_dict(mapped_state, strict=True)

    # Verify that tied weights still share the same parameter.
    if model.config.tie_word_embeddings:
        assert (
            model.lm_head.weight is model.embed_tokens.weight
        )

    print(f"Mapped tensors: {len(used_source_keys)}")
    print(f"Ignored local RoPE buffers: {len(ignored_source_keys)}")
    print("All destination tensors loaded successfully.")


In [74]:
model = SmolLMForCausalLM(config)
load_hf_weights(model, hf_model)

model.eval()

Mapped tensors: 273
Ignored local RoPE buffers: 0
All destination tensors loaded successfully.


SmolLMForCausalLM(
  (embed_tokens): Embedding(49152, 576)
  (layers): ModuleList(
    (0-29): 30 x DecoderBlock(
      (input_layernorm): RMSNorm()
      (self_attn): GroupedQueryAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=False)
        (k_proj): Linear(in_features=576, out_features=192, bias=False)
        (v_proj): Linear(in_features=576, out_features=192, bias=False)
        (o_proj): Linear(in_features=576, out_features=576, bias=False)
        (rotary_emb): RotaryEmbedding()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): SwiGLU(
        (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
        (up_proj): Linear(in_features=576, out_features=1536, bias=False)
        (down_proj): Linear(in_features=1536, out_features=576, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=576, out_features=49152, bias=False)
)

In [75]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)
print("Special tokens:", tokenizer.special_tokens_map)

Vocabulary size: 49152
Model max length: 8192
Special tokens: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}


In [76]:
text = "Transformers learn useful representations of language."

input_ids = tokenizer.encode(
    text,
    add_special_tokens=False,
)

decoded_text = tokenizer.decode(input_ids)

print("Original:", text)
print("Token IDs:", input_ids)
print("Decoded:", decoded_text)

Original: Transformers learn useful representations of language.
Token IDs: [31806, 366, 835, 3465, 11605, 282, 1789, 30]
Decoded: Transformers learn useful representations of language.


In [77]:
assert decoded_text == text
print("Tokenizer round trip passed.")

Tokenizer round trip passed.


In [79]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False,
)

input_ids = inputs["input_ids"]

print(input_ids.shape)

with torch.no_grad():
    logits = model(input_ids)

print(logits.shape)

torch.Size([1, 8])
torch.Size([1, 8, 49152])


In [80]:

@torch.inference_mode()
def generate_greedy(
    model,
    input_ids,
    max_new_tokens=30,
    eos_token_id=None,
):
    model.eval()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        output = model(generated)

        # Support both custom models and Hugging Face models
        logits = output.logits if hasattr(output, "logits") else output

        next_token = logits[:, -1, :].argmax(
            dim=-1, keepdim=True
        )

        generated = torch.cat([generated, next_token], dim=-1)

        if eos_token_id is not None:
            if (next_token == eos_token_id).all():
                break

    return generated


In [81]:

prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)

custom_ids = generate_greedy(
    model,
    input_ids,
    max_new_tokens=30,
    eos_token_id=tokenizer.eos_token_id,
)

hf_ids = generate_greedy(
    hf_model,
    input_ids,
    max_new_tokens=30,
    eos_token_id=tokenizer.eos_token_id,
)

print("Custom model:", tokenizer.decode(custom_ids[0], skip_special_tokens=True))
print("Hugging Face:", tokenizer.decode(hf_ids[0], skip_special_tokens=True))
print("Exact token match:", torch.equal(custom_ids, hf_ids))


Custom model: The capital of France is the capital of the country.

The capital of France is the capital of the country.

The capital of France is the capital of the
Hugging Face: The capital of France is the capital of the country.

The capital of France is the capital of the country.

The capital of France is the capital of the
Exact token match: True


In [93]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by a strictly positive temperature."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")

    return logits / temperature


In [95]:
def apply_top_k(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Keep the k highest logits and mask all remaining logits."""
    vocab_size = logits.shape[-1]

    if not 1 <= k <= vocab_size:
        raise ValueError(f"k must be between 1 and {vocab_size}")

    top_k_values, _ = torch.topk(logits, k, dim=-1)
    threshold = top_k_values[..., -1, None]

    return logits.masked_fill(logits < threshold, float("-inf"))


In [97]:
def apply_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """Keep the smallest high-probability token set reaching top_p."""
    if not 0.0 < top_p <= 1.0:
        raise ValueError("top_p must be greater than 0 and at most 1")

    sorted_logits, sorted_indices = torch.sort(
        logits, descending=True, dim=-1
    )

    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Initially remove tokens whose cumulative probability exceeds top_p.
    sorted_remove = cumulative_probs > top_p

    # Keep the token that crosses the threshold as well.
    sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
    sorted_remove[..., 0] = False

    # Restore the original vocabulary ordering.
    remove_mask = torch.zeros_like(sorted_remove)
    remove_mask.scatter_(-1, sorted_indices, sorted_remove)

    return logits.masked_fill(remove_mask, float("-inf"))


In [99]:
print("Temperature scaling:")
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

device = next(model.parameters()).device
input_ids = input_ids.to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

for temperature in (0.5, 1.0, 2.0):
    scaled_logits = apply_temperature(logits, temperature)
    probabilities = torch.softmax(scaled_logits, dim=-1)

    top_probs, top_ids = probabilities[0].topk(5)

    print(f"\nTemperature: {temperature}")
    for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
        token = tokenizer.decode([token_id])
        print(f"  {token!r:20} {probability:.6f}")

print("\nTop-k sampling:")
prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

k = 5
filtered_logits = apply_top_k(logits, k)
probabilities = torch.softmax(filtered_logits, dim=-1)

print("Tokens remaining:", torch.isfinite(filtered_logits).sum().item())
print("Probability sum:", probabilities.sum().item())

top_probs, top_ids = probabilities[0].topk(k)

for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20} {probability:.6f}")


print("\nTop-p sampling:")
prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

top_p = 0.9
filtered_logits = apply_top_p(logits, top_p)
probabilities = torch.softmax(filtered_logits, dim=-1)

print("Tokens retained:", torch.isfinite(filtered_logits).sum().item())
print("Probability sum:", probabilities.sum().item())

top_probs, top_ids = probabilities[0].topk(5)

for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20} {probability:.6f}")




Temperature scaling:

Temperature: 0.5
  ' the'               0.771823
  ' Paris'             0.099259
  ' located'           0.060194
  ' called'            0.021693
  ' a'                 0.017330

Temperature: 1.0
  ' the'               0.261682
  ' Paris'             0.093842
  ' located'           0.073079
  ' called'            0.043871
  ' a'                 0.039211

Temperature: 2.0
  ' the'               0.012274
  ' Paris'             0.007350
  ' located'           0.006486
  ' called'            0.005026
  ' a'                 0.004751

Top-k sampling:
Tokens remaining: 5
Probability sum: 1.0
' the'               0.511412
' Paris'             0.183399
' located'           0.142820
' called'            0.085738
' a'                 0.076631

Top-p sampling:
Tokens retained: 406
Probability sum: 1.0
' the'               0.290708
' Paris'             0.104252
' located'           0.081185
' called'            0.048737
' a'                 0.043560


In [100]:

import torch

@torch.inference_mode()
def generate_sample(
    model,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    eos_token_id: int | None = None,
    seed: int | None = None,
) -> torch.Tensor:
    """Generate tokens using temperature-scaled, filtered sampling."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens cannot be negative")

    device = input_ids.device
    generator = None

    if seed is not None:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

    model.eval()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        logits = model(generated)[:, -1, :]
        logits = apply_temperature(logits, temperature)

        if top_k is not None:
            logits = apply_top_k(logits, top_k)

        if top_p is not None:
            logits = apply_top_p(logits, top_p)

        probabilities = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        )

        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None:
            if (next_token == eos_token_id).all():
                break

    return generated


In [101]:

prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

generation_args = {
    "max_new_tokens": 30,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.9,
    "eos_token_id": tokenizer.eos_token_id,
    "seed": 42,
}

sample_a = generate_sample(model, input_ids, **generation_args)
sample_b = generate_sample(model, input_ids, **generation_args)

print("Same token sequence:", torch.equal(sample_a, sample_b))
print("\nGenerated text:")
print(tokenizer.decode(sample_a[0], skip_special_tokens=True))


Same token sequence: True

Generated text:
The capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.



In [82]:

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)

model.eval()
hf_model.eval()

with torch.inference_mode():
    custom_logits = model(input_ids)[:, -1, :]
    hf_logits = hf_model(input_ids).logits[:, -1, :]

custom_id = custom_logits.argmax(dim=-1).item()
hf_id = hf_logits.argmax(dim=-1).item()

print("Custom next token:", repr(tokenizer.decode([custom_id])))
print("HF next token:    ", repr(tokenizer.decode([hf_id])))
print("Custom token ID:", custom_id)
print("HF token ID:    ", hf_id)

# Compare the top 5 candidates from each model.
for name, logits in [
    ("Custom", custom_logits),
    ("HF", hf_logits),
]:
    top_ids = logits[0].topk(5).indices.tolist()
    print(f"\n{name} top 5:")
    for token_id in top_ids:
        print(
            f"  id={token_id:>5} "
            f"token={tokenizer.decode([token_id])!r} "
            f"logit={logits[0, token_id].item():.5f}"
        )


Custom next token: ' the'
HF next token:     ' the'
Custom token ID: 260
HF token ID:     260

Custom top 5:
  id=  260 token=' the' logit=14.02282
  id= 7042 token=' Paris' logit=12.99731
  id= 3807 token=' located' logit=12.74723
  id= 1217 token=' called' logit=12.23694
  id=  253 token=' a' logit=12.12465

HF top 5:
  id=  260 token=' the' logit=14.02282
  id= 7042 token=' Paris' logit=12.99731
  id= 3807 token=' located' logit=12.74723
  id= 1217 token=' called' logit=12.23694
  id=  253 token=' a' logit=12.12465


In [83]:

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)

custom_ids = input_ids.clone()
hf_ids = input_ids.clone()

model.eval()
hf_model.eval()

with torch.inference_mode():
    for step in range(30):
        custom_logits = model(custom_ids)[:, -1, :]
        hf_logits = hf_model(hf_ids).logits[:, -1, :]

        custom_next = custom_logits.argmax(dim=-1, keepdim=True)
        hf_next = hf_logits.argmax(dim=-1, keepdim=True)

        custom_token_id = custom_next.item()
        hf_token_id = hf_next.item()

        print(
            f"Step {step + 1:02d} | "
            f"Custom: {tokenizer.decode([custom_token_id])!r} "
            f"(ID {custom_token_id}) | "
            f"HF: {tokenizer.decode([hf_token_id])!r} "
            f"(ID {hf_token_id})"
        )

        if custom_token_id != hf_token_id:
            print("First token divergence.")
            break

        custom_ids = torch.cat([custom_ids, custom_next], dim=1)
        hf_ids = torch.cat([hf_ids, hf_next], dim=1)

        if custom_token_id == tokenizer.eos_token_id:
            print("Both models generated EOS.")
            break


Step 01 | Custom: ' the' (ID 260) | HF: ' the' (ID 260)
Step 02 | Custom: ' capital' (ID 3575) | HF: ' capital' (ID 3575)
Step 03 | Custom: ' of' (ID 282) | HF: ' of' (ID 282)
Step 04 | Custom: ' the' (ID 260) | HF: ' the' (ID 260)
Step 05 | Custom: ' country' (ID 1798) | HF: ' country' (ID 1798)
Step 06 | Custom: '.' (ID 30) | HF: '.' (ID 30)
Step 07 | Custom: '\n' (ID 198) | HF: '\n' (ID 198)
Step 08 | Custom: '\n' (ID 198) | HF: '\n' (ID 198)
Step 09 | Custom: 'The' (ID 504) | HF: 'The' (ID 504)
Step 10 | Custom: ' capital' (ID 3575) | HF: ' capital' (ID 3575)
Step 11 | Custom: ' of' (ID 282) | HF: ' of' (ID 282)
Step 12 | Custom: ' France' (ID 4649) | HF: ' France' (ID 4649)
Step 13 | Custom: ' is' (ID 314) | HF: ' is' (ID 314)
Step 14 | Custom: ' the' (ID 260) | HF: ' the' (ID 260)
Step 15 | Custom: ' capital' (ID 3575) | HF: ' capital' (ID 3575)
Step 16 | Custom: ' of' (ID 282) | HF: ' of' (ID 282)
Step 17 | Custom: ' the' (ID 260) | HF: ' the' (ID 260)
Step 18 | Custom: ' countr

In [84]:

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)

# Append the three tokens both models generated identically.
shared_ids = torch.cat(
    [
        input_ids,
        torch.tensor([[260, 3575, 282]], device=device),
    ],
    dim=1,
)

model.eval()
hf_model.eval()

with torch.inference_mode():
    custom_logits = model(shared_ids)[0, -1, :]
    hf_logits = hf_model(shared_ids).logits[0, -1, :]

diff = (custom_logits - hf_logits).abs()

print("Shared context:", repr(tokenizer.decode(shared_ids[0])))
print(f"Max logit difference:  {diff.max().item():.8f}")
print(f"Mean logit difference: {diff.mean().item():.8f}")

for name, logits in [("Custom", custom_logits), ("HF", hf_logits)]:
    print(f"\n{name} top 5:")
    for token_id in logits.topk(5).indices.tolist():
        print(
            f"ID {token_id:>5} | "
            f"{tokenizer.decode([token_id])!r} | "
            f"logit={logits[token_id].item():.5f}"
        )


Shared context: 'The capital of France is the capital of'
Max logit difference:  0.00000000
Mean logit difference: 0.00000000

Custom top 5:
ID   260 | ' the' | logit=16.90825
ID  4649 | ' France' | logit=16.13102
ID   253 | ' a' | logit=14.42898
ID   624 | ' its' | logit=14.12210
ID   511 | ' all' | logit=13.68731

HF top 5:
ID   260 | ' the' | logit=16.90825
ID  4649 | ' France' | logit=16.13102
ID   253 | ' a' | logit=14.42898
ID   624 | ' its' | logit=14.12210
ID   511 | ' all' | logit=13.68731


In [85]:

def capture_output(storage, name):
    def hook(module, inputs, output):
        # Hugging Face attention returns a tuple; our modules return tensors.
        if isinstance(output, tuple):
            output = output[0]
        storage[name] = output.detach().float().clone()
    return hook


custom_block = model.layers[0]
hf_block = hf_model.model.layers[0]

custom_captured = {}
hf_captured = {}
handles = []

for name in [
    "input_layernorm",
    "self_attn",
    "post_attention_layernorm",
    "mlp",
]:
    handles.append(
        getattr(custom_block, name).register_forward_hook(
            capture_output(custom_captured, name)
        )
    )
    handles.append(
        getattr(hf_block, name).register_forward_hook(
            capture_output(hf_captured, name)
        )
    )

try:
    with torch.inference_mode():
        _ = model(input_ids)
        _ = hf_model(input_ids)
finally:
    for handle in handles:
        handle.remove()

for name in [
    "input_layernorm",
    "self_attn",
    "post_attention_layernorm",
    "mlp",
]:
    custom_out = custom_captured[name]
    hf_out = hf_captured[name]
    diff = (custom_out - hf_out).abs()

    print(
        f"{name:25s} "
        f"max={diff.max().item():.8f}, "
        f"mean={diff.mean().item():.8f}"
    )


input_layernorm           max=0.00000000, mean=0.00000000
self_attn                 max=0.00000000, mean=0.00000000
post_attention_layernorm  max=0.00000000, mean=0.00000000
mlp                       max=0.00000000, mean=0.00000000


In [86]:

custom_block = model.layers[0]
hf_block = hf_model.model.layers[0]

prompt = "The capital of France is"
input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()
hf_model.eval()

with torch.inference_mode():
    # Obtain the exact same normalized input from the HF block.
    hf_hidden = hf_model.model.embed_tokens(input_ids)
    hf_normalized = hf_block.input_layernorm(hf_hidden)

    # Use identical input for both MLP implementations.
    custom_gate = custom_block.mlp.gate_proj(hf_normalized)
    hf_gate = hf_block.mlp.gate_proj(hf_normalized)

    custom_up = custom_block.mlp.up_proj(hf_normalized)
    hf_up = hf_block.mlp.up_proj(hf_normalized)

    custom_activated = torch.nn.functional.silu(custom_gate)
    hf_activated = torch.nn.functional.silu(hf_gate)

    custom_product = custom_activated * custom_up
    hf_product = hf_activated * hf_up

    custom_mlp_out = custom_block.mlp.down_proj(custom_product)
    hf_mlp_out = hf_block.mlp.down_proj(hf_product)

for name, a, b in [
    ("gate projection", custom_gate, hf_gate),
    ("up projection", custom_up, hf_up),
    ("SiLU(gate)", custom_activated, hf_activated),
    ("gate * up", custom_product, hf_product),
    ("down projection / MLP output", custom_mlp_out, hf_mlp_out),
]:
    diff = (a - b).abs()
    print(
        f"{name:30s} "
        f"max={diff.max().item():.8f}, "
        f"mean={diff.mean().item():.8f}"
    )


gate projection                max=0.00000000, mean=0.00000000
up projection                  max=0.00000000, mean=0.00000000
SiLU(gate)                     max=0.00000000, mean=0.00000000
gate * up                      max=0.00000000, mean=0.00000000
down projection / MLP output   max=0.00000000, mean=0.00000000


In [87]:

custom_block = model.layers[0]
hf_block = hf_model.model.layers[0]

prompt = "The capital of France is"
input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()
hf_model.eval()

with torch.inference_mode():
    # 1. Feed identical embeddings into both first-block attention paths.
    hidden = hf_model.model.embed_tokens(input_ids)

    custom_norm = custom_block.input_layernorm(hidden)
    hf_norm = hf_block.input_layernorm(hidden)

    custom_attn = custom_block.self_attn
    hf_attn = hf_block.self_attn

    batch_size, seq_len, _ = hidden.shape

    # 2. Compare raw Q, K, V projections before reshaping.
    custom_q = custom_attn.q_proj(custom_norm)
    hf_q = hf_attn.q_proj(hf_norm)

    custom_k = custom_attn.k_proj(custom_norm)
    hf_k = hf_attn.k_proj(hf_norm)

    custom_v = custom_attn.v_proj(custom_norm)
    hf_v = hf_attn.v_proj(hf_norm)

    for name, a, b in [
        ("Q projection", custom_q, hf_q),
        ("K projection", custom_k, hf_k),
        ("V projection", custom_v, hf_v),
    ]:
        diff = (a.float() - b.float()).abs()
        print(
            f"{name:15s} max={diff.max().item():.8f}, "
            f"mean={diff.mean().item():.8f}"
        )

    # 3. Reshape projections into attention-head format.
    # Custom implementation dimensions.
    custom_q = custom_q.view(
        batch_size,
        seq_len,
        custom_attn.num_heads,
        custom_attn.head_dim,
    ).transpose(1, 2)

    custom_k = custom_k.view(
        batch_size,
        seq_len,
        custom_attn.num_kv_heads,
        custom_attn.head_dim,
    ).transpose(1, 2)

    # Hugging Face dimensions from the model configuration.
    hf_config = hf_model.config

    hf_num_heads = hf_config.num_attention_heads
    hf_num_kv_heads = hf_config.num_key_value_heads
    hf_head_dim = hf_config.head_dim

    hf_q = hf_q.view(
        batch_size,
        seq_len,
        hf_num_heads,
        hf_head_dim,
    ).transpose(1, 2)

    hf_k = hf_k.view(
        batch_size,
        seq_len,
        hf_num_kv_heads,
        hf_head_dim,
    ).transpose(1, 2)

    # 4. Obtain Hugging Face rotary embeddings using identical position IDs.
    position_ids = torch.arange(
        seq_len, device=device
    ).unsqueeze(0)

    hf_cos, hf_sin = hf_attn.rotary_emb(
        hf_q, position_ids
    )

    # 5. Obtain custom rotary embeddings.
    custom_cos, custom_sin = custom_attn.rotary_emb(
        custom_q
    )

    # 6. Apply RoPE to custom Q and K.
    custom_q_rot, custom_k_rot = apply_rope(
        custom_q,
        custom_k,
        custom_cos,
        custom_sin,
    )

    # 7. Apply RoPE using Hugging Face's helper.
    from transformers.models.llama.modeling_llama import (
        apply_rotary_pos_emb,
    )

    hf_q_rot, hf_k_rot = apply_rotary_pos_emb(
        hf_q,
        hf_k,
        hf_cos,
        hf_sin,
    )

    # 8. Compare rotary embeddings and rotated projections.
    for name, a, b in [
        ("RoPE cosine", custom_cos, hf_cos),
        ("RoPE sine", custom_sin, hf_sin),
        ("Rotated Q", custom_q_rot, hf_q_rot),
        ("Rotated K", custom_k_rot, hf_k_rot),
    ]:
        diff = (a.float() - b.float()).abs()
        print(
            f"{name:15s} max={diff.max().item():.8f}, "
            f"mean={diff.mean().item():.8f}"
        )


Q projection    max=0.00000000, mean=0.00000000
K projection    max=0.00000000, mean=0.00000000
V projection    max=0.00000000, mean=0.00000000


AttributeError: 'LlamaAttention' object has no attribute 'rotary_emb'

In [88]:
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

custom_block = model.layers[0]
hf_block = hf_model.model.layers[0]

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

with torch.inference_mode():
    hidden = hf_model.model.embed_tokens(input_ids)
    hidden = hf_block.input_layernorm(hidden)

    custom_attn = custom_block.self_attn
    hf_attn = hf_block.self_attn

    batch_size, seq_len, _ = hidden.shape

    q = custom_attn.q_proj(hidden).view(
        batch_size, seq_len,
        custom_attn.num_heads, custom_attn.head_dim
    ).transpose(1, 2)

    k = custom_attn.k_proj(hidden).view(
        batch_size, seq_len,
        custom_attn.num_kv_heads, custom_attn.head_dim
    ).transpose(1, 2)

    position_ids = torch.arange(
        seq_len, device=device
    ).unsqueeze(0)

    custom_cos, custom_sin = custom_attn.rotary_emb(q)

    # Your Transformers version computes RoPE at the model level.
    hf_cos, hf_sin = hf_model.model.rotary_emb(hidden, position_ids)

    custom_q_rot, custom_k_rot = apply_rope(
        q, k, custom_cos, custom_sin
    )

    hf_q_rot, hf_k_rot = apply_rotary_pos_emb(
        q, k, hf_cos, hf_sin
    )

    for name, a, b in [
        ("RoPE cosine", custom_cos, hf_cos),
        ("RoPE sine", custom_sin, hf_sin),
        ("Rotated Q", custom_q_rot, hf_q_rot),
        ("Rotated K", custom_k_rot, hf_k_rot),
    ]:
        diff = (a.float() - b.float()).abs()
        print(
            f"{name:15s} max={diff.max().item():.8f}, "
            f"mean={diff.mean().item():.8f}"
        )

RoPE cosine     max=0.00000000, mean=0.00000000
RoPE sine       max=0.00000000, mean=0.00000000
Rotated Q       max=0.00000000, mean=0.00000000
Rotated K       max=0.00000000, mean=0.00000000


In [89]:
hf_block = hf_model.model.layers[0]

print("Decoder block:", type(hf_block).__name__)
print("Attention module:", type(hf_block.self_attn).__name__)
print("Attention attributes:")
print([name for name in vars(hf_block.self_attn) if "rotary" in name.lower()])
print("Model rotary attributes:")
print([name for name in vars(hf_model.model) if "rotary" in name.lower()])

Decoder block: LlamaDecoderLayer
Attention module: LlamaAttention
Attention attributes:
[]
Model rotary attributes:
[]


In [90]:
import inspect

print("Decoder forward:")
print(inspect.signature(hf_model.model.layers[0].forward))

print("\nModel forward:")
print(inspect.signature(hf_model.model.forward))

print("\nRoPE-related modules:")
for name, module in hf_model.named_modules():
    if "rotary" in name.lower() or "rotary" in type(module).__name__.lower():
        print(name, type(module).__name__)

Decoder forward:
(hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, use_cache: bool | None = False, position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None, **kwargs: *<class 'transformers.utils.generic.TransformersKwargs'>) -> torch.Tensor

Model forward:
(input_ids: torch.LongTensor | None = None, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, use_cache: bool | None = None, **kwargs: *<class 'transformers.utils.generic.TransformersKwargs'>) -> transformers.modeling_outputs.BaseModelOutputWithPast

RoPE-related modules:
model.rotary_emb LlamaRotaryEmbedding


In [91]:
hf_config_dict = hf_model.config.to_dict()

print("RoPE-related HF config values:")
for key, value in hf_config_dict.items():
    if "rope" in key.lower() or "position" in key.lower():
        print(f"{key}: {value}")

print("\nHF rotary embedding:")
print(hf_model.model.rotary_emb)

print("\nHF inv_freq (first 8):")
print(hf_model.model.rotary_emb.inv_freq[:8])

print("\nCustom rotary embedding:")
custom_rope = model.layers[0].self_attn.rotary_emb
print("theta:", custom_rope.theta)
print("inv_freq (first 8):")
print(custom_rope.inv_freq[:8])

RoPE-related HF config values:
max_position_embeddings: 8192
rope_parameters: {'rope_theta': 100000, 'rope_type': 'default'}
rope_interleaved: False

HF rotary embedding:
LlamaRotaryEmbedding()

HF inv_freq (first 8):
tensor([1.0000, 0.6978, 0.4870, 0.3398, 0.2371, 0.1655, 0.1155, 0.0806])

Custom rotary embedding:
theta: 100000.0
inv_freq (first 8):
tensor([1.0000, 0.6978, 0.4870, 0.3398, 0.2371, 0.1655, 0.1155, 0.0806])
